En este cuaderno vamos a usar un modelo ViT (Vision Transformers). El elegido es: *google/vit-base-patch16-224-in21k*

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, Image, Features, Value
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate

# 1. Fase de Entrenamiento

## 1.1 Defiendo las rutas y el modelo con el que vamos a trabajar

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes.csv"
CSV_TRAIN_MASTER_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"

OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned"
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

In [ ]:
print("Cargando el dataset de imágenes y las plantillas de texto...")
df_imagenes = pd.read_csv(CSV_IMAGENES)
df_train_text = pd.read_csv(CSV_TRAIN_MASTER_TEXT)
df_test_text = pd.read_csv(CSV_TEST_TEXT)

Cargando el dataset de imágenes y las plantillas de texto...


## 1.2 Particionado

**OJO** con este paso, tendremos que tener en cuenta el contenido de la columna *id_EXIST* porque estaríamos falseando los resultados si de un mismo vídeo tenemos un frame en el conjunto de entrenamiento, otro frame en el conjunto de test y otro frame en el conjunto de validación.

In [ ]:
# =================================================================
# FASE 1: Alineación 80/20 Estática Multimodal
# =================================================================
# Extraemos los IDs de los vídeos para que el particionado de imágenes
# sea EXACTAMENTE idéntico al de texto.
train_master_ids = df_train_text['id_EXIST'].unique()
test_ids = df_test_text['id_EXIST'].unique()

# Filtramos las imágenes basándonos en las particiones estáticas
df_train_master = df_imagenes[df_imagenes['id_EXIST'].isin(train_master_ids)].copy()
test_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

# =================================================================
# FASE 2: División Dinámica del Train Master (90% Train / 10% Valid)
# =================================================================
# Usamos GroupShuffleSplit para asegurar que todos los fotogramas
# de un mismo vídeo caen en el MISMO bloque (evita Data Leakage)
gss_train_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)

# Separamos Train (90%) y Valid (10%)
train_idx, val_idx = next(gss_train_val.split(df_train_master, groups=df_train_master['id_EXIST']))

train_df = df_train_master.iloc[train_idx].copy()
val_df = df_train_master.iloc[val_idx].copy()

print("\n--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---")
print(f"Vídeos en Train: {train_df['id_EXIST'].nunique()} (Aprox {len(train_df)} fotogramas)")
print(train_df['label'].value_counts())

print(f"\nVídeos en Valid: {val_df['id_EXIST'].nunique()} (Aprox {len(val_df)} fotogramas)")
print(val_df['label'].value_counts())

print(f"\nVídeos en Test (Intocable): {test_df['id_EXIST'].nunique()} (Aprox {len(test_df)} fotogramas)")
print(test_df['label'].value_counts())


--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---
Vídeos en Train: 1805 (Aprox 7218 fotogramas)
label
0    3750
1    3468
Name: count, dtype: int64

Vídeos en Valid: 201 (Aprox 804 fotogramas)
label
0    428
1    376
Name: count, dtype: int64

Vídeos en Test (Intocable): 502 (Aprox 2008 fotogramas)
label
0    1044
1     964
Name: count, dtype: int64


## 1.3 Conversión a Formato `Dataset` de Hugging Face

In [ ]:
def crear_dataset(dataframe):
    # Primero creamos el dataset leyendo las rutas como texto plano desde el dataframe
    dataset = Dataset.from_pandas(dataframe)

    # Casteamos la columna de texto a tipo Image() para que lea los archivos de Drive
    dataset = dataset.cast_column("path_imagen", Image())

    # Renombramos para que el modelo lo entienda
    return dataset.rename_column("path_imagen", "image")

print("Transformando rutas en imágenes procesables...")
train_dataset = crear_dataset(train_df)
valid_dataset = crear_dataset(val_df)

Transformando rutas en imágenes procesables...


## 1.4 Procesador de Imágenes (`ViTImageProcessor`)

Esto es igual que el tokenizador de texto, pero para imágenes (redimensiona a 224x224, normaliza colores, etc.)

In [ ]:
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_images(batch):
    # Toma una lista de imágenes y las convierte en los tensores matemáticos que ViT necesita
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

# Aplicamos la transformación "al vuelo" para no saturar la RAM
train_dataset.set_transform(process_images)
valid_dataset.set_transform(process_images)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

## 1.5 Inicialización del Modelo Base ViT

In [ ]:
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Necesario porque el modelo in21k original no tiene capa de clasificación
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.

## 1.6 Métrica de Evaluación

In [ ]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# Usamos tu amado F1 Score macro
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.7 Hiperparámetros del Entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # IMPORTANTE: En visión debe ser False
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # ViT prefiere Learning Rates más bajos que los LLMs
    per_device_train_batch_size=16, # ViT es ligero, aguanta un batch de 16 en Colab
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Precisión mixta para acelerar en GPU
    report_to="none"
)

## 1.8 Entrenamiento

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento visual con ViT...")
trainer.train()

print("💾 Guardando modelo visual...")
trainer.save_model(OUTPUT_DIR + "/modelo_final")
processor.save_pretrained(OUTPUT_DIR + "/modelo_final")
print("✅ ¡Entrenamiento completado!")

🚀 Iniciando entrenamiento visual con ViT...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.698574,0.688308,0.347403,0.532338
2,0.691143,0.689215,0.352714,0.533582
3,0.687662,0.685212,0.456870,0.547264
4,0.688888,0.684709,0.514558,0.552239
5,0.680045,0.681226,0.519046,0.565920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Guardando modelo visual...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Fase de Inferencia/Evaluación

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
from tqdm import tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes.csv"
MODEL_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned/modelo_final"

print("1. Reconstruyendo las particiones exactas del entrenamiento...")
df_imagenes = pd.read_csv(CSV_IMAGENES)

1. Reconstruyendo las particiones exactas del entrenamiento...


In [5]:
# Recreamos el particionado EXACTO usando el mismo random_state=42
gss_train_test = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss_train_test.split(df_imagenes, groups=df_imagenes['id_EXIST']))
temp_df = df_imagenes.iloc[temp_idx]

gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(temp_df, groups=temp_df['id_EXIST']))
val_df = temp_df.iloc[val_idx] # Usaremos Validación para no tocar el Test aún

In [6]:
print("2. Cargando tu modelo ViT entrenado desde Drive...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = ViTImageProcessor.from_pretrained(MODEL_DIR)
model = ViTForImageClassification.from_pretrained(MODEL_DIR).to(device)
model.eval() # Modo evaluación (apaga el aprendizaje)

print(f"Iniciando inferencia sobre {val_df['id_EXIST'].nunique()} vídeos...")

# Diccionario para guardar las predicciones agrupadas por vídeo
predicciones_por_video = {}

2. Cargando tu modelo ViT entrenado desde Drive...


Loading weights:   0%|          | 0/200 [00:02<?, ?it/s]

Iniciando inferencia sobre 376 vídeos...


In [7]:
# 3. Inferencia frame a frame
for index, row in tqdm(val_df.iterrows(), total=len(val_df)):
    id_vid = row['id_EXIST']
    img_path = row['path_imagen']
    true_label = row['label']

    # Cargamos imagen
    try:
        image = Image.open(img_path).convert("RGB")
    except:
        continue # Por si alguna imagen se corrompió al guardarse

    # Preprocesamos y pasamos por el modelo
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        # Aplicamos Softmax para obtener porcentajes [Probabilidad 0, Probabilidad 1]
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        prob_misogino = probs[1].item() # Nos quedamos con la prob de que sea clase 1

    # Agrupamos en el diccionario
    if id_vid not in predicciones_por_video:
        predicciones_por_video[id_vid] = {'probs': [], 'true_label': true_label}

    predicciones_por_video[id_vid]['probs'].append(prob_misogino)

100%|██████████| 1504/1504 [21:25<00:00,  1.17it/s]


## 2.1 Max-Pooling

Con que solamente 1 solo fotograma de los 4 sea etiquetado como misógino, el video entero se marcará como misógino.

In [8]:
# 4. Estrategias de Pooling (Agregación)
y_true = []
y_pred_max = []  # Estrategia 1: Max-Pooling (Basta con que 1 frame sea machista)


for id_vid, data in predicciones_por_video.items():
    y_true.append(data['true_label'])

    probabilidades = data['probs']

    # ---------------------------------------------------------
    # ESTRATEGIA 1: Max-Pooling
    # ---------------------------------------------------------
    # Cogemos el frame donde el modelo estuvo más seguro de que era misógino
    max_prob = max(probabilidades)
    y_pred_max.append(1 if max_prob > 0.5 else 0)

In [9]:
# 5. Resultados
print("\n" + "="*50)
print("🏆 RESULTADOS A NIVEL DE VÍDEO (MAX-POOLING)")
print("=> Lógica: Si 1 fotograma es detectado como misógino, el vídeo entero lo es.")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_max, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_max):.4f}")


🏆 RESULTADOS A NIVEL DE VÍDEO (MAX-POOLING)
=> Lógica: Si 1 fotograma es detectado como misógino, el vídeo entero lo es.
F1-Score (Macro): 0.5642
Accuracy: 0.5824


# 2.2 Average Pooling

Se cogerá el porcentaje de seguridad del modelo para los 4 fotogramas y haremos la media. Si la media >= 50% se cataloga vídeo como misógino.

In [10]:
# 4. Estrategias de Pooling (Agregación)
y_true = []
y_pred_mean = [] # Estrategia 2: Mean-Pooling (Promedio de todos los frames)

for id_vid, data in predicciones_por_video.items():
    y_true.append(data['true_label'])

    probabilidades = data['probs']

    # ---------------------------------------------------------
    # ESTRATEGIA 2: Mean-Pooling
    # ---------------------------------------------------------
    # Promediamos las probabilidades de todos los frames del vídeo
    mean_prob = sum(probabilidades) / len(probabilidades)
    y_pred_mean.append(1 if mean_prob > 0.5 else 0)

In [11]:
print("\n" + "="*50)
print("📊 RESULTADOS A NIVEL DE VÍDEO (MEAN-POOLING)")
print("=> Lógica: Promedio de agresividad visual de todo el vídeo.")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_mean, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_mean):.4f}")


📊 RESULTADOS A NIVEL DE VÍDEO (MEAN-POOLING)
=> Lógica: Promedio de agresividad visual de todo el vídeo.
F1-Score (Macro): 0.5248
Accuracy: 0.5824


## 2.3 Votación por mayoría

Hacemos que al menos 2 o 3 fotogramas sean clasificados como misóginos para considerar el vídeo como misógino.

In [12]:
# 4. Estrategias de Pooling (Agregación)
y_true = []
y_pred_majority = [] # Estrategia 3: Votación por Mayoría

for id_vid, data in predicciones_por_video.items():
    y_true.append(data['true_label'])

    probabilidades = data['probs']
    # Convertimos las probabilidades a predicciones binarias (0 o 1) para cada frame
    predicciones_binarias = [1 if p > 0.5 else 0 for p in probabilidades]

    # ---------------------------------------------------------
    # ESTRATEGIA 3: Votación por Mayoría (Majority Vote)
    # ---------------------------------------------------------
    # Contamos cuántos frames se predijeron como misóginos
    votos_misogino = sum(predicciones_binarias)
    mitad_frames = len(predicciones_binarias) / 2

    # Lógica: Si hay más de la mitad de votos misóginos, es misógino.
    # Si hay EMPATE (ej: 2 vs 2), decidimos clasificarlo como misógino (ante la duda, penalizamos la agresión)
    if votos_misogino >= mitad_frames:
        y_pred_majority.append(1)
    else:
        y_pred_majority.append(0)

In [13]:
print("\n" + "="*50)
print("🗳️ RESULTADOS A NIVEL DE VÍDEO (MAJORITY VOTE)")
print("=> Lógica: Gana la clase que más fotogramas tenga (en caso de empate, gana Misógino).")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_majority, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_majority):.4f}")
print("\nMatriz de Confusión (Majority Vote):\n", confusion_matrix(y_true, y_pred_majority))
print("\nClassification Report (Majority Vote):\n", classification_report(y_true, y_pred_majority))


🗳️ RESULTADOS A NIVEL DE VÍDEO (MAJORITY VOTE)
=> Lógica: Gana la clase que más fotogramas tenga (en caso de empate, gana Misógino).
F1-Score (Macro): 0.5238
Accuracy: 0.5691

Matriz de Confusión (Majority Vote):
 [[165  41]
 [121  49]]

Classification Report (Majority Vote):
               precision    recall  f1-score   support

           0       0.58      0.80      0.67       206
           1       0.54      0.29      0.38       170

    accuracy                           0.57       376
   macro avg       0.56      0.54      0.52       376
weighted avg       0.56      0.57      0.54       376

